# 05 SVHN DNN 優化：LeakyReLU + Learning Rate Schedule

前一階段已使用 `04_svhn_dnn_tensorflow.ipynb` 建立 Simple Dense DNN。該模型保留 `512 -> 256 -> 128` 的 Dense 架構，使用 ReLU、batch size 128 與固定 learning rate，作為第一個深度學習比較基準。

本階段保留相同的 Dense layer 寬度，但調整訓練策略：

- 將 ReLU 改為 LeakyReLU。
- 將 batch size 從 128 改為 64。
- 加入 `ReduceLROnPlateau`，當 validation accuracy 停滯時自動降低 learning rate。
- 保留 EarlyStopping，避免訓練無止境進行。

這些調整的目標，是觀察 Advanced Dense DNN 是否能在同一份 SVHN 子集上，比 Simple Dense DNN 取得更高的 test accuracy。

> 注意：本階段仍然是 Dense DNN，尚未使用 CNN。即使進階優化後接近 70%，模型仍然沒有直接利用影像空間結構；下一份 notebook 會以 CNN 展示影像模型的主流做法。


## 0. 匯入套件與設定

為了和 `03`、`04` 公平比較，本階段使用同樣的 SVHN 子集設定。


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from sklearn.metrics import accuracy_score

import tensorflow as tf

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

TRAIN_SAMPLES = 12000
TEST_SAMPLES = 3000
EPOCHS = 20
BATCH_SIZE = 128


## 1. 下載與讀取 SVHN

若已經跑過前面的 SVHN notebook，Colab 會使用快取。


In [ ]:
SVHN_BASE_URL = 'http://ufldl.stanford.edu/housenumbers/'

train_path = tf.keras.utils.get_file(
    'svhn_train_32x32.mat',
    origin=SVHN_BASE_URL + 'train_32x32.mat',
    cache_dir='.',
    cache_subdir='data/svhn'
)

test_path = tf.keras.utils.get_file(
    'svhn_test_32x32.mat',
    origin=SVHN_BASE_URL + 'test_32x32.mat',
    cache_dir='.',
    cache_subdir='data/svhn'
)

print('train_path:', train_path)
print('test_path:', test_path)


In [ ]:
def load_svhn_mat(path):
    data = loadmat(path)
    X = np.transpose(data['X'], (3, 0, 1, 2))
    y = data['y'].reshape(-1) % 10
    return X, y.astype('int64')


def stratified_subset(X, y, n_samples, random_state=42):
    rng = np.random.default_rng(random_state)
    classes = np.unique(y)
    base = n_samples // len(classes)
    remainder = n_samples % len(classes)

    selected = []
    for rank, cls in enumerate(classes):
        cls_idx = np.where(y == cls)[0]
        take = base + (1 if rank < remainder else 0)
        take = min(take, len(cls_idx))
        selected.append(rng.choice(cls_idx, size=take, replace=False))

    selected = np.concatenate(selected)
    rng.shuffle(selected)
    return X[selected], y[selected]


X_train_full, y_train_full = load_svhn_mat(train_path)
X_test_full, y_test_full = load_svhn_mat(test_path)

X_train, y_train = stratified_subset(X_train_full, y_train_full, TRAIN_SAMPLES, RANDOM_STATE)
X_test, y_test = stratified_subset(X_test_full, y_test_full, TEST_SAMPLES, RANDOM_STATE + 1)

X_train_float = X_train.astype('float32') / 255.0
X_test_float = X_test.astype('float32') / 255.0

print('X_train_float:', X_train_float.shape)
print('X_test_float:', X_test_float.shape)


## 2. 進階優化方向

Simple Dense DNN 已經證明：在真實街景數字資料上，神經網路可以明顯改善 raw-pixel 傳統 ML 的限制。本階段進一步加入三個優化技巧，觀察模型是否能從約 65% 再推進到接近 70%。

| 優化技巧 | 說明 | 預期效果 |
| --- | --- | --- |
| LeakyReLU | 當輸入小於 0 時仍保留一點斜率 | 降低部分神經元長期沒有更新的風險 |
| batch size 64 | 每次更新使用較少樣本 | 增加更新次數，可能讓訓練更細緻 |
| ReduceLROnPlateau | validation accuracy 停滯時降低 learning rate | 讓模型在後期用較小步伐繼續微調 |

實作流程如下：

```text
建模 -> 設定參數 -> 執行訓練 -> 評估模型 -> 與 Simple Dense DNN 比較
```


## 2.1 建立 Advanced Dense DNN

此模型保留與 `04` 相同的 `512 -> 256 -> 128` Dense 寬度，但將 activation 從 ReLU 改為 LeakyReLU。這樣可以讓小於 0 的輸入仍保留微小梯度，而不是完全歸零。


In [ ]:
def add_leaky_dense_block(model, units, negative_slope=0.1):
    model.add(tf.keras.layers.Dense(units))
    model.add(tf.keras.layers.LeakyReLU(negative_slope=negative_slope))


def build_advanced_dense_dnn():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(32, 32, 3)))
    model.add(tf.keras.layers.Flatten())
    add_leaky_dense_block(model, 512)
    add_leaky_dense_block(model, 256)
    add_leaky_dense_block(model, 128)
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)
advanced_model = build_advanced_dense_dnn()
advanced_model.summary()


## 2.2 設定訓練參數

本階段最多訓練 `120` 個 epochs，但同時使用 EarlyStopping 與 ReduceLROnPlateau。當 validation accuracy 停滯時，ReduceLROnPlateau 會降低 learning rate，讓模型用較小步伐繼續搜尋更好的權重。


In [ ]:
advanced_params = {
    'model': 'Advanced Dense DNN (LeakyReLU + LR schedule)',
    'hidden_units': '512 / 256 / 128',
    'activation': 'leaky_relu',
    'learning_rate': 0.001,
    'batch_size': 64,
    'epochs': 120,
    'early_stopping_patience': 15,
    'reduce_lr_patience': 3,
    'reduce_lr_factor': 0.5,
    'min_lr': 1e-5,
}

advanced_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=advanced_params['early_stopping_patience'],
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        mode='max',
        factor=advanced_params['reduce_lr_factor'],
        patience=advanced_params['reduce_lr_patience'],
        min_lr=advanced_params['min_lr']
    )
]

display(pd.DataFrame([advanced_params]))


## 2.3 執行訓練

這一組訓練時間會比 Simple Dense DNN 長，因為 batch size 較小、最多 epochs 較多，而且模型會在 validation accuracy 停滯時動態降低 learning rate。


In [ ]:
start = time.perf_counter()
advanced_history = advanced_model.fit(
    X_train_float,
    y_train,
    validation_split=0.2,
    epochs=advanced_params['epochs'],
    batch_size=advanced_params['batch_size'],
    callbacks=advanced_callbacks,
    verbose=1
)
advanced_training_time = time.perf_counter() - start
print(f'training_time_sec: {advanced_training_time:.2f}')


## 2.4 評估模型

模型訓練完成後，同時回報 train accuracy 與 test accuracy。若 test accuracy 接近 0.70，表示進階優化確實讓 Dense DNN 往前推進；若 train accuracy 明顯高於 test accuracy，則代表模型仍有過擬合或資料表示限制。


In [ ]:
advanced_train_loss, advanced_train_acc = advanced_model.evaluate(
    X_train_float, y_train, verbose=0
)
advanced_test_loss, advanced_test_acc = advanced_model.evaluate(
    X_test_float, y_test, verbose=0
)

advanced_result = {
    **advanced_params,
    'epochs_ran': len(advanced_history.history['loss']),
    'best_val_acc': max(advanced_history.history['val_accuracy']),
    'train_acc': advanced_train_acc,
    'test_acc': advanced_test_acc,
    'train_test_gap': advanced_train_acc - advanced_test_acc,
    'training_time_sec': advanced_training_time,
    'history': advanced_history.history,
}

advanced_df = pd.DataFrame([
    {k: v for k, v in advanced_result.items() if k != 'history'}
])
display(advanced_df)



### 2.4.1 評估結果觀察

進階優化的目標，是讓 test accuracy 從 Simple Dense DNN 的水準再往上推進。若 test accuracy 提升但 train/test gap 也變大，代表模型能力提升的同時也要注意過擬合。


## 2.5 目前 SVHN 模型排名

以下表格使用同一份 SVHN cropped 子集進行比較：train 12,000 筆、test 3,000 筆。分數是課程目前固定的實測參考值；若重新執行 notebook，TensorFlow 模型分數可能有小幅浮動。

| 排名 | 方法 | Train Accuracy | Test Accuracy | 觀察 |
| ---: | --- | ---: | ---: | --- |
| 1 | Advanced Dense DNN：LeakyReLU + ReduceLROnPlateau + batch size 64 | 0.8501 | 0.6973 | 進階訓練策略讓 Dense DNN 接近 70%，但 train/test gap 也變大。 |
| 2 | Simple Dense DNN | 0.7476 | 0.6540 | 第一個深度學習基準，已明顯優於 RBF SVM。 |
| 3 | RBF SVM raw pixels + StandardScaler | 0.6947 | 0.5170 | 傳統 ML 在真實影像 raw pixels 上測試表現受限。 |

目前排名顯示，神經網路優化可以進一步提升 SVHN 數字分類表現。不過 Advanced Dense DNN 仍然使用 flatten 後的一維向量，尚未利用影像的空間結構，因此下一步會進入 CNN。


## 3. 觀察訓練曲線

訓練曲線可用來判斷模型是否只是 train accuracy 變高，或 validation accuracy 也同步改善。若 train accuracy 持續上升但 validation accuracy 停滯，通常代表模型可能開始過度貼合訓練資料。


In [ ]:
history_df = pd.DataFrame(advanced_history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df[['loss', 'val_loss']].plot(ax=axes[0])
axes[0].set_title('Advanced Dense DNN loss')
axes[0].set_xlabel('Epoch')

history_df[['accuracy', 'val_accuracy']].plot(ax=axes[1])
axes[1].set_title('Advanced Dense DNN accuracy')
axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.show()


## 4. 和前面 SVHN 模型比較

下表只比較同一份 SVHN train 12,000 / test 3,000 子集上的結果。不同資料集的分數不放入同一張排行榜。


In [ ]:
previous_results = pd.DataFrame([
    {
        'model': 'RBF SVM raw pixels + StandardScaler',
        'train_acc': 0.6947,
        'test_acc': 0.5170,
        'training_time_sec': 91.65,
    },
    {
        'model': 'Simple Dense DNN',
        'train_acc': 0.7476,
        'test_acc': 0.6540,
        'training_time_sec': 22.38,
    },
])

advanced_results = pd.DataFrame([
    {
        'model': advanced_result['model'],
        'train_acc': advanced_result['train_acc'],
        'test_acc': advanced_result['test_acc'],
        'training_time_sec': advanced_result['training_time_sec'],
    }
])

comparison_df = pd.concat([previous_results, advanced_results], ignore_index=True)
comparison_df = comparison_df.sort_values('test_acc', ascending=False)
display(comparison_df)

plt.figure(figsize=(9, 4))
plt.barh(comparison_df['model'], comparison_df['test_acc'])
plt.xlim(0, 1)
plt.xlabel('Test accuracy')
plt.title('SVHN subset: Dense DNN optimization progress')
plt.gca().invert_yaxis()
plt.show()


## 5. 為什麼 Dense DNN 仍然有極限？

從比較表可以看到，Advanced Dense DNN 透過 LeakyReLU、batch size 64 與 learning rate schedule，通常能比 Simple Dense DNN 再提升一段 test accuracy。這代表訓練策略確實會影響神經網路表現。

不過，Dense DNN 的核心限制仍然存在：它一開始就把圖片攤平成一維向量，沒有保留像素的上下左右關係。對 SVHN 這類真實照片資料而言，數字邊緣、局部筆畫、背景干擾與位置偏移都很重要；因此下一個合理方向，是使用 CNN 讓模型直接學習局部影像特徵。


## 課後練習與思考題

請根據本階段的 Advanced Dense DNN 完成下列整理。練習重點是判讀優化策略是否真的改善模型，並思考當 train score 明顯高於 test score 時，如何降低過擬合風險。

1. 觀察 Advanced Dense DNN 的 train/test gap，整理至少三種避免過擬合的方法。
2. 比較 Simple Dense DNN 與 Advanced Dense DNN 的 train/test score。
3. 整理 LeakyReLU、batch size 64 與 ReduceLROnPlateau 各自可能帶來的影響。
4. 思考為什麼 Dense DNN 經過優化後仍然需要銜接 CNN。


In [ ]:
# 練習 1 參考答案：判讀過擬合，並建立可替換的 regularized model

practice_overfit_df = comparison_df.copy()
practice_overfit_df['train_test_gap'] = (
    practice_overfit_df['train_acc'] - practice_overfit_df['test_acc']
)
display(practice_overfit_df[[
    'model', 'train_acc', 'test_acc', 'train_test_gap', 'training_time_sec'
]])


def add_regularized_leaky_dense_block(model, units, dropout_rate, l2_strength=1e-4):
    model.add(tf.keras.layers.Dense(
        units,
        kernel_regularizer=tf.keras.regularizers.l2(l2_strength)
    ))
    model.add(tf.keras.layers.LeakyReLU(negative_slope=0.1))
    model.add(tf.keras.layers.Dropout(dropout_rate))


def build_regularized_dense_dnn():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(32, 32, 3)))
    # SVHN 是影像資料，可用小幅平移與縮放增加訓練樣本變化；幅度不宜過大。
    model.add(tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05))
    model.add(tf.keras.layers.RandomZoom(height_factor=0.05, width_factor=0.05))
    model.add(tf.keras.layers.Flatten())
    add_regularized_leaky_dense_block(model, 512, dropout_rate=0.30)
    add_regularized_leaky_dense_block(model, 256, dropout_rate=0.25)
    add_regularized_leaky_dense_block(model, 128, dropout_rate=0.20)
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


regularized_model = build_regularized_dense_dnn()
regularized_model.summary()


### 練習 1 參考答案：避免過擬合的方法

Advanced Dense DNN 的固定參考分數是 `train_acc=0.8501`、`test_acc=0.6973`，train/test gap 約 `0.1528`。這代表模型已經比 Simple Dense DNN 更會分類，但也更明顯地貼近訓練資料，因此需要討論降低過擬合的方法。

| 方法 | Keras 寫法提示 | 用途 |
| --- | --- | --- |
| EarlyStopping | `tf.keras.callbacks.EarlyStopping(...)` | validation 指標停止改善時提早結束訓練，避免模型繼續背訓練資料。 |
| Dropout | `tf.keras.layers.Dropout(0.2)` | 訓練時隨機關掉部分神經元，降低模型對特定路徑的依賴。 |
| L2 regularization | `kernel_regularizer=tf.keras.regularizers.l2(1e-4)` | 限制權重不要長得太大，讓決策規則更平滑。 |
| 降低模型容量 | `Dense(512 -> 256 -> 128)` 改成 `Dense(256 -> 128)` | 減少參數量，讓模型比較不容易記住訓練資料細節。 |
| Data augmentation | `RandomTranslation`、`RandomZoom` | 對影像做小幅平移或縮放，讓模型看過更多合理變化。 |

上方 code cell 提供一個可替換實驗模型：加入小幅資料增強、L2 regularization 與 Dropout，用來觀察 train/test gap 是否縮小。


In [ ]:
# 練習 2 參考答案：比較 Simple Dense DNN 與 Advanced Dense DNN

practice_compare_df = comparison_df.copy()
practice_compare_df['train_test_gap'] = practice_compare_df['train_acc'] - practice_compare_df['test_acc']
display(practice_compare_df[[
    'model', 'train_acc', 'test_acc', 'train_test_gap', 'training_time_sec'
]])



### 練習 2 參考答案：比較 Simple Dense DNN 與 Advanced Dense DNN

目前固定參考結果中，Advanced Dense DNN 的 `test_acc` 較高，表示 LeakyReLU、batch size 64 與 learning rate schedule 確實帶來改善。不過 Advanced Dense DNN 的 train/test gap 也更大，因此準確率提升的同時也要注意過擬合。


### 練習 3 參考答案：整理三個進階優化技巧

| 優化技巧 | 可能影響 |
| --- | --- |
| LeakyReLU | 小於 0 的輸入仍保留一點斜率，降低部分神經元長期沒有更新的風險。 |
| batch size 64 | 每個 epoch 中更新次數較多，訓練過程可能更細緻，但時間通常較長。 |
| ReduceLROnPlateau | validation accuracy 停滯時降低 learning rate，讓模型後期用較小步伐微調。 |

這三個技巧主要改善訓練與收斂方式；若 train/test gap 過大，還需要搭配 regularization 或資料增強。


### 練習 4 參考答案：整理 Dense DNN 仍需要 CNN 的原因

| 原因 | 說明 |
| --- | --- |
| Flatten 破壞空間結構 | 圖片被攤平成 3072 維向量後，模型不再直接知道哪些像素彼此相鄰。 |
| 影像特徵具有局部性 | 數字邊緣、筆畫與角落形狀通常是局部圖案，CNN 的卷積核更適合學這類特徵。 |
| 參數使用效率 | Dense layer 需要大量全連接權重；CNN 透過權重共享，能更有效處理影像。 |
| 泛化能力 | SVHN 有位置偏移、光線與背景變化，CNN 的影像歸納偏置通常更有利於泛化。 |

Advanced Dense DNN 可接近 70%，但影像任務若要大幅提升，下一步應使用 CNN。
